# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

**Recommended:** create a venv, then `pip install -r requirements.txt` from the repo root, and register/select a Jupyter kernel for that venv (see `context/ENVIRONMENT_SETUP.md`).

Optional: the cell below uses `uv` and shell paths that target **Linux/macOS** (and `wget`). On **Windows**, prefer `requirements.txt` + the correct kernel instead of running the install cell.

> **After first install, restart the kernel** so packages (especially `torch`, `vllm` if installed) are picked up.

### Optional install cell (Linux/macOS)

Uncomment commands in the next cell if you use `uv`; otherwise rely on `pip install -r requirements.txt`.

In [1]:
# Install deps into the *same* Python as this notebook (avoids wrong paths on Windows / wrong cwd).
# Optional `uv` lines at bottom: Linux/macOS only. On Windows use sys.executable or .venv\Scripts\python.exe.
from pathlib import Path
import sys


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


ROOT = _repo_root()
REQ = ROOT / "requirements.txt"

print("Repo root:", ROOT)
print("requirements.txt:", REQ, "| exists:", REQ.is_file())
print("This kernel's Python:", sys.executable)
print()

if REQ.is_file():
    print("Install (copy into a terminal, or set RUN_PIP_INSTALL_HERE = True below):")
    print(f'  "{sys.executable}" -m pip install -U pip')
    print(f'  "{sys.executable}" -m pip install -r "{REQ}"')
    print()
    print("Register kernel (after venv is active / using the Python you want):")
    print(f'  "{sys.executable}" -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"')
else:
    print("Could not find requirements.txt next to judger.py — check repo layout.")

# One-shot install into *this* Jupyter kernel (same as running the pip lines above)
RUN_PIP_INSTALL_HERE = False
if RUN_PIP_INSTALL_HERE and REQ.is_file():
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQ)])
    print("Done. Restart the kernel, then continue.")

# Linux/macOS only (optional):
# !wget -qO- https://astral.sh/uv/install.sh | sh
# !uv venv .venv --seed
# !uv pip install -r requirements.txt

Repo root: /mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm
requirements.txt: /mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm/requirements.txt | exists: True
This kernel's Python: /home/sardo/cse151b-venv/bin/python

Install (copy into a terminal, or set RUN_PIP_INSTALL_HERE = True below):
  "/home/sardo/cse151b-venv/bin/python" -m pip install -U pip
  "/home/sardo/cse151b-venv/bin/python" -m pip install -r "/mnt/c/Users/sardo/OneDrive/Desktop/Classes/projects/math-qa-llm/requirements.txt"

Register kernel (after venv is active / using the Python you want):
  "/home/sardo/cse151b-venv/bin/python" -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"


### Kernel note

Shell activation (`source .venv/...`) does **not** change which Python Jupyter uses. Pick the project venv **kernel** instead.

In [2]:
# No-op: Jupyter runs the interpreter of the *selected kernel*, not the shell you activate here.
print("Using:", __import__("sys").executable)

Using: /home/sardo/cse151b-venv/bin/python


In [ ]:
# ── Google Colab Setup ──────────────────────────────────────────────────────
# This cell is a no-op when running locally.  On Colab it installs the
# required packages and mounts Google Drive so checkpoints survive across
# sessions.  Run it first, then restart the runtime if prompted.
import sys, importlib

try:
    import google.colab          # noqa: F401
    _IS_COLAB = True
except ImportError:
    _IS_COLAB = False

if _IS_COLAB:
    print("Colab detected — installing packages (takes ~3-5 min)...")
    import subprocess
    pkgs = [
        "vllm>=0.6.0",
        "transformers>=4.45.0",
        "bitsandbytes>=0.43.0",
        "peft>=0.12.0",
        "trl>=0.12.0",
        "datasets",
        "accelerate>=0.34.0",
        "sympy>=1.12",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
    print("Packages installed. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
    print()
    print("Next: upload public.jsonl and private.jsonl to")
    print("  MyDrive/math-qa-llm/data/raw/  on your Google Drive.")
    print("Then continue running cells below.")
else:
    print("Running locally — skipping Colab setup.")


## 2. Imports & Configuration

Paths are resolved from the repo root (folder containing `judger.py`), so the notebook works whether your **current working directory** is the repo root or `notebooks/`.

- `DATA_PATH` — default `data/raw/public.jsonl` (labeled)
- `PRIVATE_PATH` — `data/raw/private.jsonl` (switch `DATA_PATH` to this for submission-only runs; no local accuracy)
- `OUTPUT_PATH` — default under `artifacts/logs/runs/`
- `N_QUESTIONS` — `None` = full file; integer = smoke-test subset (must match generation + scoring)
- `TEST_RANDOM_SUBSET` — if `True` and `N_QUESTIONS` is an int, draw a **random** sample (see `RANDOM_SEED`) instead of the **first** N rows — better for quick sanity checks across the file
- `GPU_ID` — passed to `CUDA_VISIBLE_DEVICES` (single GPU is usually `"0"`)
- `MAX_TOKENS` — max **new** tokens per answer; Qwen3-Thinking needs headroom for long `<think>` chains before `\boxed{}` (default **8192**; **16384** safer on hard items, slower)

In [ ]:
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Optional


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for d in (p, *p.parents):
        if (d / "judger.py").is_file():
            return d
    return p


REPO_ROOT = repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"

PUBLIC_PATH  = REPO_ROOT / "data" / "raw" / "public.jsonl"
PRIVATE_PATH = REPO_ROOT / "data" / "raw" / "private.jsonl"

DATA_MODE = "public"
DATA_PATH = PUBLIC_PATH if DATA_MODE == "public" else PRIVATE_PATH

N_QUESTIONS        = None   # None = all questions; set to 5 for smoke test
TEST_RANDOM_SUBSET = False
RANDOM_SEED        = 42

RUN_NAME        = f"adaptive_{DATA_MODE}_v2"
OUTPUT_PATH     = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_results.jsonl"
CHECKPOINT_PATH = REPO_ROOT / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_checkpoint.jsonl"

# ── Phase token budgets (tuned for HF Transformers on A30 24GB BF16) ─────────
PHASE1_THINKING_BUDGET    = 1024   # was 4096 — 4× speedup; adequate for first-pass
PHASE1_MAX_TOKENS         = 4096   # was 6144
PHASE1_N_SAMPLES          = 1

PHASE2_THINKING_BUDGET    = 4096   # keep full budget for uncertain questions
PHASE2_MAX_TOKENS         = 5120   # was 6144
PHASE2_N_SAMPLES          = 3      # auto-bumped to 8 in EBM load cell if verifier found
PHASE2_TEMPERATURE        = 0.65
PHASE2_REPETITION_PENALTY = 1.05

MAX_TOKENS = PHASE2_MAX_TOKENS

# ── Transformers batching ─────────────────────────────────────────────────────
CHUNK_SIZE = 6   # sequences per model.generate() call (safe for A30 24GB BF16)

# ── EBM verifier config ───────────────────────────────────────────────────────
# Trained by notebooks/06_train_ebm_verifier.ipynb after the full public inference run.
# Uses the SAME Qwen3-4B backbone (llm) — zero extra VRAM for weights (~10 KB head only).
# Falls back to majority vote gracefully if head file not found.
USE_EBM  = True
EBM_PATH = REPO_ROOT / "artifacts" / "models" / "ebm_verifier_v1"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from transformers import AutoTokenizer

# ── vLLM disabled — DSMLP CUDA 12.8 vs vLLM 0.21.0 compiled for CUDA 13 ─────
VLLM_AVAILABLE = False
LLM            = None
SamplingParams = None
USE_VLLM       = False
print("vLLM disabled — using HuggingFace Transformers (DSMLP CUDA 12.8 incompatibility).")

from tqdm.auto import tqdm

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    DRIVE_BASE      = Path('/content/drive/MyDrive/math-qa-llm')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    PUBLIC_PATH     = DRIVE_BASE / "data" / "raw" / "public.jsonl"
    PRIVATE_PATH    = DRIVE_BASE / "data" / "raw" / "private.jsonl"
    DATA_PATH       = PUBLIC_PATH if DATA_MODE == "public" else PRIVATE_PATH
    OUTPUT_PATH     = DRIVE_BASE / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_results.jsonl"
    CHECKPOINT_PATH = DRIVE_BASE / "artifacts" / "logs" / "runs" / f"{RUN_NAME}_checkpoint.jsonl"
    EBM_PATH        = DRIVE_BASE / "artifacts" / "models" / "ebm_verifier_v1"
    (DRIVE_BASE / "artifacts" / "logs" / "runs").mkdir(parents=True, exist_ok=True)

print("IS_COLAB   :", IS_COLAB)
print("REPO_ROOT  :", REPO_ROOT)
print("DATA_MODE  :", DATA_MODE)
print("DATA_PATH  :", DATA_PATH, "| exists:", DATA_PATH.is_file())
print("N_QUESTIONS:", N_QUESTIONS)
print("RUN_NAME   :", RUN_NAME)
print("CHUNK_SIZE :", CHUNK_SIZE)
print("USE_EBM    :", USE_EBM)
print("vLLM       :", VLLM_AVAILABLE)


## 3. Load the Dataset

Data is **JSONL**: one JSON object per line. Paths in this notebook default to **`data/raw/public.jsonl`** (development) and **`data/raw/private.jsonl`** (submission); only the public file includes labels.

### Fields (by split)

| Field | Public (`public.jsonl`) | Private (`private.jsonl`) |
|------|-------------------------|---------------------------|
| `id` | Integer, unique per problem | Same |
| `question` | Problem text, **LaTeX**; free-form items use **`[ANS]`** placeholders where answers go | Same |
| `options` | Present for **MCQ** (list of LaTeX choices); omitted for **free-form** | Same rule |
| `answer` | **Present**: MCQ → single capital letter string (e.g. `"C"`); free-form → **list of strings** (one entry per `[ANS]`, in order) | **Omitted** (withheld for evaluation) |

### Formats

- **Free-form:** One or more answers; every sub-answer must match for the problem to count as correct when scoring locally.
- **MCQ:** Model should pick one letter matching the labeled option.

Use **`DATA_PATH`** in the config cell to point at public vs private; run scoring (§7) only when every loaded row has an **`answer`** field.

In [ ]:
with open(DATA_PATH, encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = len(data) - n_mcq
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

if N_QUESTIONS is None:
    data_run = data
else:
    k = min(int(N_QUESTIONS), len(data))
    if TEST_RANDOM_SUBSET:
        rng      = random.Random(RANDOM_SEED)
        data_run = rng.sample(data, k=k)
    else:
        data_run = data[:k]

_nrun_mcq = sum(bool(d.get("options")) for d in data_run)
print(f"Running on {len(data_run)} questions  ({_nrun_mcq} MCQ, {len(data_run)-_nrun_mcq} free-form)")


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics, "
    "from algebra and calculus to number theory and combinatorics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Identify what is given and what you need to find.\n"
    "2. PLAN: Write down the key equations, formulas, or theorems you will use.\n"
    "3. SOLVE: Work through each step carefully. Compute intermediate results explicitly. "
    "Pay special attention to arithmetic - do not skip steps.\n"
    "4. VERIFY: Check that your answer satisfies all conditions in the problem. "
    "Check units, sign, and order of magnitude.\n"
    "5. ANSWER: Put your final answer in \\boxed{}.\n\n"
    "Additional rules:\n"
    "- If the problem has multiple blanks ([ANS] placeholders), put ALL answers "
    "comma-separated in ONE \\boxed{} in the order they appear. "
    "Example: \\boxed{3, -7, 42}.\n"
    "- Simplify all fractions and radical expressions completely.\n"
    "- You'd better be sure of your answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician with deep knowledge of all areas of mathematics. "
    "This problem is very important to my career - please think carefully and be precise.\n\n"
    "Solve using this structured approach:\n"
    "1. UNDERSTAND: Read the problem and all answer choices carefully.\n"
    "2. PLAN: Identify the relevant concepts, formulas, or theorems that apply.\n"
    "3. SOLVE: Work through the problem step by step. Compute intermediate results "
    "explicitly - do not skip arithmetic steps.\n"
    "4. ELIMINATE: Cross out answer choices that are clearly wrong.\n"
    "5. VERIFY: Confirm your chosen answer is consistent with every condition in the problem.\n"
    "6. ANSWER: On the very last line of your response, write ONLY \\boxed{X} "
    "where X is the letter of the correct answer (A-J). "
    "Do not write any text after \\boxed{}.\n\n"
    "You'd better be sure of your answer."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    n_ans = question.count("[ANS]")
    if n_ans > 1:
        hint = (
            f"\n\n[Note: This problem has {n_ans} answers. "
            f"Put all {n_ans} answers comma-separated in ONE \\boxed{{}} "
            f"in the order they appear in the question.]"
        )
        return SYSTEM_PROMPT_MATH, question + hint
    return SYSTEM_PROMPT_MATH, question


## 5. Load Model

Loads **Qwen3-4B-Thinking-2507** with HuggingFace Transformers.

**vLLM is disabled** — vLLM 0.21.0 (the only Python 3.13 wheel on PyPI) was compiled against CUDA 13 and requires ELF-versioned symbols (`libcudart.so.13`) not present in DSMLP's CUDA 12.8. patchelf is unavailable.

GPU auto-detection picks the right precision tier:
- **≥ 20 GB VRAM** (A30, A100): BF16 + PyTorch built-in SDPA — no extra packages needed, no quantization loss
- **< 20 GB VRAM** (RTX 3070, etc.): NF4 4-bit via BitsAndBytes — fits in 8 GB with minor accuracy tradeoff

`tokenizer.padding_side = "left"` is set here — required for correct batched generation with decoder-only models.

In [ ]:
import torch
import torch.nn as nn
from collections import Counter
from math import ceil


def extract_boxed(text: str) -> str:
    """Extract last \\boxed{...} content (nested braces supported)."""
    matches = []
    needle  = r"\boxed{"
    i = 0
    while i < len(text):
        idx = text.find(needle, i)
        if idx == -1:
            break
        j, depth, start = idx + len(needle), 1, idx + len(needle)
        while j < len(text) and depth:
            if text[j] == "{":
                depth += 1
            elif text[j] == "}":
                depth -= 1
                if depth == 0:
                    matches.append(text[start:j])
                    break
            j += 1
        i = idx + 1
    return matches[-1].strip() if matches else ""


def strip_thinking(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def is_uncertain(response: str, finish_reason: str = "") -> bool:
    if "length" in str(finish_reason).lower():
        return True
    if not extract_boxed(response):
        return True
    if len(strip_thinking(response)) < 30:
        return True
    return False


def choose_best_sample(samples: list, finish_reasons: list) -> dict:
    """Majority vote; tie-break by longest trace."""
    extracted = [extract_boxed(s) for s in samples]
    nonempty  = [e for e in extracted if e]
    if nonempty:
        counts       = Counter(nonempty)
        top_count    = counts.most_common(1)[0][1]
        tied_answers = {ans for ans, cnt in counts.items() if cnt == top_count}
        candidates   = [i for i, ans in enumerate(extracted) if ans in tied_answers]
        best_idx     = max(candidates, key=lambda i: len(samples[i]))
        best_answer  = extracted[best_idx]
    else:
        top_count, best_idx, best_answer = 0, 0, ""
    uncertain = (
        is_uncertain(samples[best_idx], finish_reasons[best_idx])
        or top_count < ceil(len(samples) / 2)
    )
    return {
        "response":        samples[best_idx],
        "answer":          best_answer,
        "finish_reason":   finish_reasons[best_idx],
        "consensus_count": top_count,
        "n_samples":       len(samples),
        "uncertain":       uncertain,
    }


def make_sampling_params(max_tokens: int, temperature: float = 0.6,
                          repetition_penalty: float = 1.0) -> dict:
    """Return model.generate() kwargs dict — replaces vLLM SamplingParams."""
    return dict(
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        top_k=20,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )


def generate_batch(prompts: list, gen_kwargs: dict, chunk_size: int = CHUNK_SIZE) -> list:
    """Batch-generate using HF Transformers with left-padding and crash-safe chunking."""
    results = []
    for i in range(0, len(prompts), chunk_size):
        chunk  = prompts[i : i + chunk_size]
        inputs = tokenizer(
            chunk, return_tensors="pt", padding=True,
            truncation=True, max_length=16384,
        ).to(llm.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out_ids = llm.generate(**inputs, **gen_kwargs)
        new_ids = out_ids[:, prompt_len:]
        for seq in new_ids:
            last_tok = seq[-1].item()
            finish   = "length" if last_tok != tokenizer.eos_token_id else "stop"
            text     = tokenizer.decode(seq, skip_special_tokens=True).strip()
            results.append({"text": text, "finish_reason": finish})
        del inputs, out_ids, new_ids
        torch.cuda.empty_cache()
    return results


def build_chat_prompt(item: dict, thinking_budget=None,
                       prefix: str = "", suffix: str = "") -> str:
    """Render a chat prompt string. Falls back to plain-text budget hint if needed."""
    system, user = build_prompt(item["question"], item.get("options"))
    if prefix:
        user = prefix + user
    if suffix:
        user = user + suffix
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    kwargs = dict(tokenize=False, add_generation_prompt=True, enable_thinking=True)
    if thinking_budget is not None:
        kwargs["thinking_budget"] = thinking_budget
    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        if thinking_budget is None:
            raise
        hint = (
            f"Use at most about {thinking_budget} thinking tokens. "
            "Be concise but do not skip necessary arithmetic.\n\n"
        )
        messages[1]["content"] = hint + messages[1]["content"]
        kwargs.pop("thinking_budget", None)
        return tokenizer.apply_chat_template(messages, **kwargs)


def load_checkpoint(path) -> dict:
    if not path.exists():
        return {}
    records = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            records[str(rec["id"])] = rec
    print(f"Checkpoint: {len(records)} records from {path.name}")
    return records


def write_checkpoint(path, records: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for rec in sorted(records.values(), key=lambda r: int(r["id"])):
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


# ── EBM verifier helpers ──────────────────────────────────────────────────────
# Defined here so they're available before the EBM load cell runs.
# Both functions degrade gracefully to majority vote when verifier_head is None.

class VerifierHead(nn.Module):
    """Tiny linear scoring head on top of the frozen Qwen3-4B backbone (llm).

    Uses last-token pooling: the hidden state of the final non-padding token in the
    (question + candidate solution) sequence is projected to a scalar score.
    Higher score = more likely to be a correct solution.

    Backbone is never modified — only this head's ~2,561 parameters train.
    """
    def __init__(self, hidden_size: int):
        super().__init__()
        self.head = nn.Linear(hidden_size, 1, bias=True)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def score(self, base_model, input_ids, attention_mask) -> torch.Tensor:
        """Forward pass returning (batch,) scalar scores. No gradients on base_model."""
        with torch.no_grad():
            hidden = base_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
            ).hidden_states[-1]                          # (batch, seq_len, hidden)
        # Last-token pooling: index of last real (non-padding) token per sequence
        seq_lengths       = attention_mask.sum(dim=1) - 1   # (batch,)
        last_token_hidden = hidden[
            torch.arange(hidden.size(0), device=hidden.device),
            seq_lengths,
        ]                                                    # (batch, hidden)
        return self.head(last_token_hidden).squeeze(-1)      # (batch,)


def score_candidates_ebm(item: dict, candidates: list, verifier_head) -> list:
    """Score all candidates in ONE forward pass using VerifierHead + frozen llm.

    Uses `tokenizer` and `llm` already in scope — no separate model or tokenizer.
    Temporarily switches padding to right (needed for last-token pooling),
    then restores left-padding for subsequent generation calls.

    Returns list of float scores (higher = more likely correct).
    """
    _system = "You are evaluating whether this mathematical solution is correct and complete."
    _user   = item["question"]
    if item.get("options"):
        _labels = [chr(65 + i) for i in range(len(item["options"]))]
        _user  += "\n\nOptions:\n" + "\n".join(
            f"{l}. {o.strip()}" for l, o in zip(_labels, item["options"])
        )
    texts = []
    for cand in candidates:
        msgs = [
            {"role": "system",    "content": _system},
            {"role": "user",      "content": _user},
            {"role": "assistant", "content": cand},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False))

    # Right-padding for classification (last real token = full-sequence representation)
    _saved_side            = tokenizer.padding_side
    tokenizer.padding_side = "right"
    enc = tokenizer(
        texts, padding=True, truncation=True,
        max_length=4096, return_tensors="pt",
    ).to(llm.device)
    tokenizer.padding_side = _saved_side   # restore left-padding for generation

    scores = verifier_head.score(llm, enc["input_ids"], enc["attention_mask"])
    return scores.float().tolist()


def choose_best_sample_ebm(
    item: dict,
    samples: list,
    finish_reasons: list,
    verifier_head=None,
) -> dict:
    """Choose best sample via EBM reranking; falls back to majority vote if head is None.

    EBM path: scores all N candidates holistically, picks highest-scored one.
    Safety: if EBM's pick has no \\boxed{} but other candidates do, defers to majority vote.
    All returned dicts include 'ebm_used' and 'ebm_scores' for downstream analysis.
    """
    if verifier_head is not None:
        scores   = score_candidates_ebm(item, samples, verifier_head)
        best_idx = scores.index(max(scores))
        best_ans = extract_boxed(samples[best_idx])

        # Safety fallback: EBM picked something with no \boxed{}, but others have it
        if not best_ans and any(extract_boxed(s) for s in samples):
            result = choose_best_sample(samples, finish_reasons)
            result.update({"ebm_used": False, "ebm_fallback": True, "ebm_scores": scores})
            return result

        finish = finish_reasons[best_idx]
        return {
            "response":        samples[best_idx],
            "answer":          best_ans,
            "finish_reason":   finish,
            "consensus_count": 1,
            "n_samples":       len(samples),
            "uncertain":       is_uncertain(samples[best_idx], finish),
            "ebm_used":        True,
            "ebm_fallback":    False,
            "ebm_scores":      scores,
            "ebm_best_score":  scores[best_idx],
        }
    else:
        result = choose_best_sample(samples, finish_reasons)
        result.update({"ebm_used": False, "ebm_fallback": False, "ebm_scores": None})
        return result


In [ ]:
# ── Load model with HuggingFace Transformers (A30 24GB: BF16 + SDPA) ─────────
# vLLM disabled due to CUDA 12.8 / vLLM 0.21.0 ELF symbol version incompatibility.
# Using SDPA (PyTorch built-in scaled dot-product attention) — no flash_attn package needed.
from transformers import AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "left"   # CRITICAL: left-pad for batched decoder-only generation

_vram_gb  = (torch.cuda.get_device_properties(0).total_memory / 1e9
             if torch.cuda.is_available() else 0)
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU: {_gpu_name} | VRAM: {_vram_gb:.1f} GB")

if _vram_gb >= 20:
    # A30 24 GB / A100 — BF16 + SDPA (PyTorch built-in, no flash_attn package needed)
    _mkw    = dict(dtype=torch.bfloat16, device_map="auto",
                   trust_remote_code=True, attn_implementation="sdpa")
    _qlabel = "BF16+SDPA"
else:
    # Small GPU (<20 GB) — NF4 4-bit quantization fallback
    from transformers import BitsAndBytesConfig
    _mkw    = dict(
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        ),
        device_map="auto", trust_remote_code=True, attn_implementation="sdpa",
    )
    _qlabel = "NF4+SDPA"

llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_mkw)
llm.eval()
print(f"Model loaded ({_qlabel}) | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

# VLLM: if not VLLM_AVAILABLE:
# VLLM:     raise RuntimeError("vLLM unavailable ...")
# VLLM: llm = LLM(model=MODEL_ID, quantization=_quant, load_format=_load_fmt,
# VLLM:           gpu_memory_utilization=_gpu_util, max_model_len=16384,
# VLLM:           trust_remote_code=True, max_num_seqs=_max_seqs,
# VLLM:           max_num_batched_tokens=_batch_tok)


In [ ]:
# ── Load EBM verifier head ────────────────────────────────────────────────────
# Trained by notebooks/06_train_ebm_verifier.ipynb (run after full public inference).
# Backbone = llm already loaded above — ZERO additional VRAM for weights (~10 KB head).
# If head file not found: verifier_head=None → choose_best_sample_ebm() falls back
# to majority vote automatically. No code changes needed to switch modes.

verifier_head = None

if USE_EBM and (EBM_PATH / "verifier_head.pt").is_file():
    try:
        _vcfg = json.loads((EBM_PATH / "verifier_config.json").read_text())
        verifier_head = VerifierHead(llm.config.hidden_size).to(llm.device).to(torch.bfloat16)
        verifier_head.load_state_dict(
            torch.load(EBM_PATH / "verifier_head.pt", map_location=llm.device)
        )
        verifier_head.eval()
        print(f"EBM verifier head loaded ({_vcfg.get('type', 'ebm_v1')}) | "
              f"VRAM still {torch.cuda.memory_allocated()/1e9:.2f} GB (head is ~10 KB)")
        # More candidates = better EBM reranking — bump N when EBM is active
        if PHASE2_N_SAMPLES < 8:
            PHASE2_N_SAMPLES = 8
            print(f"PHASE2_N_SAMPLES bumped to {PHASE2_N_SAMPLES} "
                  "(EBM ranks all 8 candidates; majority vote only needed 3)")
    except Exception as _e:
        print(f"EBM load failed ({_e}) — falling back to majority vote")
        verifier_head = None
else:
    print("EBM verifier head not found — using majority vote "
          "(train it with 06_train_ebm_verifier.ipynb after the full public run)")


### Legacy: simple single-question Transformers path (disabled)

The cell below is the original unbatched loop — one `model.generate()` call per question, no checkpointing, no adaptive phases. **Commented out.** The active generation path is in Section 6.

In [8]:
# --- Transformers load path (disabled when using vLLM — do not run both; GPU OOM) ---
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"  # required for correct batched generation in decoder-only LMs
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
#
# _device = next(llm.parameters()).device
# print("Model device:", _device)


## 6. Generate Responses — Adaptive Two-Phase

Concentrates compute where it matters most:

1. **Phase 1 — fast full sweep:** every question gets one response with a short thinking budget (`thinking_budget=PHASE1_THINKING_BUDGET`, `max_new_tokens=PHASE1_MAX_TOKENS`). Prompts are batched in chunks of `CHUNK_SIZE=6` and checkpointed after every chunk — if the pod dies, the run resumes from where it left off.

2. **Phase 2 — targeted retry:** questions flagged as uncertain (truncated, missing `\boxed{}`, or too short after `<think>`) are retried with a full thinking budget and `PHASE2_N_SAMPLES=3` majority-vote samples. Each question is checkpointed immediately after so no work is lost mid-retry.

**Uncertainty detection** (`is_uncertain()`): a response is uncertain if the finish reason is `"length"` (model hit `max_new_tokens`), no `\boxed{}` was found, or the answer section after `</think>` is under 30 characters.

### Active: chunked batch generation with crash-safe checkpointing

In [ ]:
# ── Adaptive two-phase generation (HuggingFace Transformers path) ─────────────
# Phase 1: one fast response per question (thinking_budget=PHASE1_THINKING_BUDGET).
# Phase 2: retry uncertain questions with N=PHASE2_N_SAMPLES majority vote (or EBM reranking).
# Checkpoints after EVERY chunk in Phase 1 and after EVERY question in Phase 2
# so the pod can die and restart without losing progress.

response_records = load_checkpoint(CHECKPOINT_PATH)

# ── Phase 1 ────────────────────────────────────────────────────────────────────
phase1_params  = make_sampling_params(PHASE1_MAX_TOKENS, temperature=0.6)
missing_phase1 = [item for item in data_run if str(item["id"]) not in response_records]
print(f"Phase 1: {len(missing_phase1)} missing / {len(data_run)} total "
      f"({len(data_run) - len(missing_phase1)} checkpointed)")

with tqdm(total=len(missing_phase1), desc="Phase 1", unit="q") as pbar:
    for chunk_start in range(0, len(missing_phase1), CHUNK_SIZE):
        batch   = missing_phase1[chunk_start : chunk_start + CHUNK_SIZE]
        prompts = [build_chat_prompt(item, thinking_budget=PHASE1_THINKING_BUDGET)
                   for item in batch]
        outputs = generate_batch(prompts, phase1_params, chunk_size=len(batch))

        for item, out in zip(batch, outputs):
            response = out["text"]
            finish   = out["finish_reason"]
            response_records[str(item["id"])] = {
                "id":              item["id"],
                "phase_used":      1,
                "response":        response,
                "answer":          extract_boxed(response),
                "finish_reason":   finish,
                "uncertain":       is_uncertain(response, finish),
                "n_samples":       1,
                "consensus_count": 1,
                "ebm_used":        False,   # Phase 1 never uses EBM; marks record type clearly
            }
        write_checkpoint(CHECKPOINT_PATH, response_records)   # every chunk — crash safe
        pbar.update(len(batch))

p1_uncertain = sum(1 for item in data_run
                   if response_records.get(str(item["id"]), {}).get("uncertain"))
print(f"Phase 1 done: {p1_uncertain}/{len(data_run)} uncertain")

# ── Phase 2 ────────────────────────────────────────────────────────────────────
phase2_params    = make_sampling_params(PHASE2_MAX_TOKENS, temperature=PHASE2_TEMPERATURE,
                                         repetition_penalty=PHASE2_REPETITION_PENALTY)
phase1_uncertain = [
    item for item in data_run
    if response_records.get(str(item["id"]), {}).get("uncertain")
    and int(response_records.get(str(item["id"]), {}).get("phase_used", 0)) < 2
]
RETRY_PREFIX      = "Previous attempt was unclear. Solve this again carefully from scratch:\n\n"
MCQ_VERIFY_SUFFIX = (
    "\n\nAfter finding your answer, check each option against the problem conditions. "
    "Eliminate any letter that clearly fails. "
    "Then on the very last line write ONLY \\boxed{X}."
)

if phase1_uncertain:
    print(f"Phase 2: {len(phase1_uncertain)} uncertain × {PHASE2_N_SAMPLES} samples"
          f"  [{'EBM reranking' if verifier_head is not None else 'majority vote'}]")
    for item in tqdm(phase1_uncertain, desc="Phase 2", unit="q"):
        suffix      = MCQ_VERIFY_SUFFIX if item.get("options") else ""
        prompt_text = build_chat_prompt(
            item, thinking_budget=PHASE2_THINKING_BUDGET,
            prefix=RETRY_PREFIX, suffix=suffix,
        )
        outputs        = generate_batch([prompt_text] * PHASE2_N_SAMPLES, phase2_params)
        samples        = [o["text"]          for o in outputs]
        finish_reasons = [o["finish_reason"] for o in outputs]

        # Retrieve Phase 1 record so we can archive it in the Phase 2 record
        # (used later by 06_train_ebm_verifier.ipynb as natural wrong/right training pairs)
        _p1_rec = response_records.get(str(item["id"]), {})

        chosen = choose_best_sample_ebm(
            item=item,
            samples=samples,
            finish_reasons=finish_reasons,
            verifier_head=verifier_head,   # None → falls back to majority vote automatically
        )
        chosen.update({
            "id":              item["id"],
            "phase_used":      2,
            "phase1_response": _p1_rec.get("response", ""),   # archived for EBM training (nb06)
            "phase1_answer":   _p1_rec.get("answer",   ""),
        })
        response_records[str(item["id"])] = chosen
        write_checkpoint(CHECKPOINT_PATH, response_records)   # every question — crash safe

# ── Final assembly ─────────────────────────────────────────────────────────────
missing_ids = [item["id"] for item in data_run if str(item["id"]) not in response_records]
if missing_ids:
    raise RuntimeError(f"Missing responses for {len(missing_ids)} id(s): {missing_ids[:10]}")

responses       = [response_records[str(item["id"])]["response"] for item in data_run]
phase_counts    = Counter(response_records[str(item["id"])].get("phase_used") for item in data_run)
uncertain_count = sum(bool(response_records[str(item["id"])].get("uncertain")) for item in data_run)
finish_counts   = Counter(str(response_records[str(item["id"])].get("finish_reason")) for item in data_run)
ebm_count       = sum(bool(response_records[str(item["id"])].get("ebm_used")) for item in data_run)

print(f"\nDone: {len(responses)} responses | "
      f"phase1={phase_counts.get(1,0)} phase2={phase_counts.get(2,0)} "
      f"uncertain={uncertain_count} ebm_used={ebm_count}")
print(f"Finish reasons: {dict(finish_counts)}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

for i in range(min(2, len(responses))):
    rec = response_records[str(data_run[i]["id"])]
    print(f"\n── Response {i} (id={data_run[i].get('id')}, phase={rec.get('phase_used')}) ──")
    print(responses[i][:500], "..." if len(responses[i]) > 500 else "")

# VLLM: phase1_outputs = llm.generate(phase1_prompts, phase1_params)
# VLLM: for item, out in zip(missing_phase1, phase1_outputs):
# VLLM:     response = out.outputs[0].text.strip()
# VLLM:     finish_reason = str(out.outputs[0].finish_reason)
# VLLM: phase2_outputs_flat = llm.generate(phase2_prompts, phase2_params)
# VLLM: samples = [phase2_outputs_flat[j].outputs[0].text.strip() for j in range(start, end)]
# VLLM: finish_reasons = [str(phase2_outputs_flat[j].outputs[0].finish_reason) for j in range(start, end)]


### Legacy: simple unbatched loop (disabled)

Original single-question loop with no checkpointing. Commented out — superseded by the adaptive two-phase cell above.

In [10]:
# --- Transformers generation path (disabled when using vLLM above — do not run both) ---
# prompts = []
# for item in data_run:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=True,
#     )
#     prompts.append(prompt_text)
#
# responses = []
# print(
#     f"Generating {len(prompts)} response(s), max_new_tokens={MAX_TOKENS} each "
#     "(raise MAX_TOKENS in config if answers get cut off).",
#     flush=True,
# )
#
# with torch.no_grad():
#     for prompt_text in tqdm(prompts, desc="Generate", unit="q"):
#         inputs = tokenizer(
#             prompt_text,
#             return_tensors="pt",
#             truncation=True,
#             max_length=16384,
#         ).to(_device)
#         output_ids = llm.generate(
#             **inputs,
#             max_new_tokens=MAX_TOKENS,
#             temperature=0.6,
#             top_p=0.95,
#             top_k=20,
#             min_p=0.0,
#             repetition_penalty=1.0,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#         new_tokens = output_ids[0, inputs["input_ids"].shape[1] :]
#         responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
#
# print(f"Done. len(responses)={len(responses)} (expected {len(data_run)}).", flush=True)
#
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data_run[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")


## 7. Score Responses

**Requires** `answer` in each row (public set). If you loaded `private.jsonl`, skip this section and only export responses.

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [11]:
print("N_QUESTIONS:", N_QUESTIONS)
print("len(data_run):", len(data_run))
print("len(responses):", len(responses) if "responses" in globals() else "missing")
print("len(results):", len(results) if "results" in globals() else "missing")
print([x["id"] for x in data_run[:10]])

N_QUESTIONS: 50
len(data_run): 50
len(responses): 50
len(results): missing
[228, 51, 563, 501, 457, 285, 209, 1116, 178, 864]


In [ ]:
import multiprocessing as _mp


def _judge_worker(pred, gold_list, repo_root_str, result_queue):
    """Runs in a separate process — can be killed if SymPy loops forever."""
    try:
        import sys
        if repo_root_str not in sys.path:
            sys.path.insert(0, repo_root_str)
        from judger import Judger
        j = Judger(strict_extract=False)
        r = j.auto_judge(pred=pred, gold=gold_list,
                         options=[[]] * len(gold_list))
        result_queue.put(bool(r))
    except Exception:
        result_queue.put(False)


def judge_with_timeout(pred, gold_list, repo_root_str, timeout=30):
    """Run auto_judge in a subprocess with a hard timeout.

    On Linux (DSMLP), process.terminate() reliably kills a stuck SymPy loop.
    Falls back to False on timeout or any exception.
    """
    q = _mp.Queue()
    p = _mp.Process(target=_judge_worker,
                    args=(pred, gold_list, repo_root_str, q))
    p.start()
    p.join(timeout=timeout)
    if p.is_alive():
        p.terminate()
        p.join()
        return False   # timed out
    try:
        return q.get_nowait()
    except Exception:
        return False


def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


if "responses" not in globals():
    raise NameError(
        "`responses` is not defined. Finish the **Generate** cell above first."
    )
if len(responses) != len(data_run):
    raise ValueError(
        f"len(responses)={len(responses)} but len(data_run)={len(data_run)}."
    )
if not all("answer" in item for item in data_run):
    raise ValueError(
        "Scoring needs an `answer` field per row. Use `public.jsonl` for section 7."
    )

_rr_str = str(REPO_ROOT)   # passed to subprocess so it can import judger

print(
    f"Scoring {len(data_run)} responses — MCQ is instant; "
    f"free-form uses a subprocess with 30-second hard timeout per question.",
    flush=True,
)

_timeout_count = 0
results = []
with tqdm(total=len(data_run), desc="Scoring", unit="q", dynamic_ncols=True) as pbar:
    for item, response in zip(data_run, responses):
        pbar.set_postfix_str(f"id={item.get('id')}", refresh=True)

        is_mcq = bool(item.get("options"))
        gold   = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            correct   = judge_with_timeout(response, gold_list, _rr_str, timeout=30)

        results.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "gold":     gold,
            "response": response,
            "correct":  correct,
        })
        pbar.update(1)

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]


def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0


print(f"MCQ       : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"Free-form : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"Overall   : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = DATA_MODE == "public"  # Private has no labels; set False automatically.

if "responses" not in globals():
    raise RuntimeError("Run Section 6 generation before saving results.")
if len(responses) != len(data_run):
    raise RuntimeError(f"responses length {len(responses)} != data_run length {len(data_run)}")

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

if SAVE_EVAL:
    if "results" not in globals():
        raise RuntimeError("SAVE_EVAL=True requires running Section 7 scoring first.")
    # Build a fast lookup from id → scoring result
    _score_by_id = {str(r["id"]): r for r in results}
    records = []
    for item in data_run:
        sid  = str(item["id"])
        r    = _score_by_id.get(sid, {})
        meta = response_records.get(sid, {}) if "response_records" in globals() else {}
        records.append({
            "id":              item["id"],
            "is_mcq":         r.get("is_mcq", bool(item.get("options"))),
            "gold":            r.get("gold", item.get("answer")),
            "response":        r.get("response", meta.get("response", "")),
            "correct":         r.get("correct", False),
            "phase_used":      meta.get("phase_used"),
            "uncertain":       meta.get("uncertain"),
            "finish_reason":   meta.get("finish_reason"),
            "consensus_count": meta.get("consensus_count"),
            "n_samples":       meta.get("n_samples"),
            # ── EBM training fields ───────────────────────────────────────────
            # phase1_response / phase1_answer are set on Phase 2 records only;
            # they let 06_train_ebm_verifier.ipynb build natural (correct, wrong) pairs
            # without needing to re-run inference.
            "phase1_response": meta.get("phase1_response", ""),
            "phase1_answer":   meta.get("phase1_answer",   ""),
            "ebm_used":        meta.get("ebm_used", False),
        })
else:
    records = []
    for item, response in zip(data_run, responses):
        meta = response_records.get(str(item["id"]), {}) if "response_records" in globals() else {}
        records.append({
            "id":              item["id"],
            "is_mcq":         bool(item.get("options")),
            "response":        response,
            "phase_used":      meta.get("phase_used"),
            "uncertain":       meta.get("uncertain"),
            "finish_reason":   meta.get("finish_reason"),
            "consensus_count": meta.get("consensus_count"),
            "n_samples":       meta.get("n_samples"),
            "phase1_response": meta.get("phase1_response", ""),
            "phase1_answer":   meta.get("phase1_answer",   ""),
            "ebm_used":        meta.get("ebm_used", False),
        })

with open(out_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

_correct_count = sum(1 for r in records if r.get("correct")) if SAVE_EVAL else "n/a"
print(f"Saved {len(records)} records to {out_path}")
if SAVE_EVAL:
    print(f"  correct={_correct_count}/{len(records)} "
          f"({100*_correct_count/len(records):.1f}%)")

## 10. Generate Submission CSV

Convert the saved JSONL into a competition-format CSV (`id,response`).
Works for both public (evaluation) runs and private (submission) runs.
The `response` column holds the **full model trace** (competition requirement).

In [ ]:
import csv
from datetime import date

SUBMISSION_INPUT  = OUTPUT_PATH
SUBMISSION_OUTPUT = (
    REPO_ROOT / "artifacts" / "submissions"
    / f"submission_{date.today().isoformat()}.csv"
)
SUBMISSION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

rows = []
with open(SUBMISSION_INPUT, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        rows.append({"id": rec["id"], "response": rec["response"]})

rows.sort(key=lambda r: r["id"])

with open(SUBMISSION_OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "response"], quoting=csv.QUOTE_ALL)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {len(rows)} rows -> {SUBMISSION_OUTPUT}")


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!